# Point of this Notebook

A similarity index over your 14-dimensional regime_vector so you can ask "which past weeks looked
like this week, and what happened next?" -> A SQL query cannot express.

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

## Prepare the source table

In [0]:
CATALOG = "raghavan_trading_signals"
REGIMES = f"{CATALOG}.gold.weekly_market_regimes"

from pyspark.sql.functions import date_format, col

# Add the PK column only if it isn't already there (re-run safe).
existing_cols = [f.name for f in spark.table(REGIMES).schema.fields]
if "week_id" not in existing_cols:
    spark.sql(f"ALTER TABLE {REGIMES} ADD COLUMNS (week_id STRING)")
    print("Added week_id column.")
else:
    print("week_id column already exists, skipping ADD COLUMNS.")

# Idempotent: re-running just re-derives the same values.
spark.sql(f"UPDATE {REGIMES} SET week_id = date_format(week_start, 'yyyy-MM-dd')")

# Idempotent: setting an already-set property is a no-op.
spark.sql(f"ALTER TABLE {REGIMES} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Verify uniqueness of the PK and report.
total = spark.table(REGIMES).count()
uniq = spark.table(REGIMES).select("week_id").distinct().count()
assert total == uniq, "week_id is not unique!"
print(f"{total} weeks, week_id unique. CDF enabled.")

## Endpoint + Delta-Sync index

In [0]:
import datetime
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)
VS_ENDPOINT = "raghavan-trading-signals-vs"
INDEX_NAME = f"{CATALOG}.gold.regime_vector_index"

try:
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print("Endpoint creating...")
except Exception as e:
    print("Endpoint exists or:", e)

vsc.wait_for_endpoint(name=VS_ENDPOINT, timeout=datetime.timedelta(minutes=30))
print("Endpoint ONLINE")

In [0]:
import datetime

existing = [i["name"] for i in vsc.list_indexes(name=VS_ENDPOINT).get("vector_indexes", [])]

if INDEX_NAME in existing:
    print("Index already exists, attaching to it.")
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
else:
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=REGIMES,
        pipeline_type="TRIGGERED",
        primary_key="week_id",
        embedding_vector_column="regime_vector",
        embedding_dimension=14,
    )
    print("Index creating. Waiting for first sync...")

index.wait_until_ready(verbose=True, timeout=datetime.timedelta(minutes=30))
print("Index READY")

## Query: find similar weeks + what happened next

In [0]:
from pyspark.sql.functions import desc

latest = (spark.table(REGIMES).orderBy(desc("week_start"))
          .select("week_id", "regime_vector").first())
query_vector = list(latest["regime_vector"])
query_week = latest["week_id"]

res = index.similarity_search(
    query_vector=query_vector,
    columns=["week_id", "avg_market_return", "weekly_vix", "pct_stocks_oversold",
             "pct_macd_bullish", "next_1w_return", "next_1w_positive"],
    num_results=6,   # 6 because #1 is the query week itself
    disable_notice=True
)

def fmt(rowv, label=""):
    wk, mret, vix, ovsld, macd, nxt, nxt_pos = rowv[:7]
    # Forward-looking cols are NULL for the most recent week (no realized outcome yet)
    if nxt is None or nxt_pos is None:
        nxt_str = "next 1w: N/A (not realized yet)"
    else:
        nxt_str = f"next 1w: {'UP' if nxt_pos == 1 else 'DOWN'} ({float(nxt):+.4f})"
    return (f"{label}{wk} | ret {float(mret):+.4f} | VIX {float(vix):.1f} | "
            f"oversold {float(ovsld)*100:4.1f}% | macd_bull {float(macd)*100:4.1f}% | "
            f"{nxt_str}")

rows = res["result"]["data_array"]

anchor = next((r for r in rows if str(r[0]) == str(query_week)), None)
print("QUERY WEEK (anchor):")
if anchor:
    print("  " + fmt(anchor))
else:
    print(f"  {query_week} not returned in results")

print(f"\nWeeks most similar to {query_week}:")
for rowv in rows:
    if str(rowv[0]) == str(query_week):
        continue
    print("  " + fmt(rowv))

## Making a Reusable function (the app calls this) 
- For the same purpose as above

In [0]:
import numpy as np

def find_similar_regimes(week_id=None, top_k=5):
    regimes = spark.table(REGIMES)
    rowq = (regimes.filter(col("week_id") == week_id).first() if week_id
            else regimes.orderBy(desc("week_start")).first())
    if not rowq:
        return {"error": f"no regime for {week_id}"}
    res = index.similarity_search(
        query_vector=list(rowq["regime_vector"]),
        columns=["week_id", "avg_market_return", "weekly_vix", "pct_stocks_oversold",
                 "pct_macd_bullish", "next_1w_return", "next_1w_positive", "next_2w_return"],
        num_results=top_k + 1,
        disable_notice=True
    )
    weeks = []
    for r in res["result"]["data_array"]:
        if str(r[0]) == str(rowq["week_id"]):
            continue
        weeks.append({"week": r[0], "market_return": float(r[1]), "vix": float(r[2]),
                      "pct_oversold": float(r[3]), "pct_macd_bullish": float(r[4]),
                      "next_1w_return": float(r[5]) if r[5] is not None else None,
                      "next_1w_positive": int(r[6]) if r[6] is not None else None})
    weeks = weeks[:top_k]
    up = sum(1 for w in weeks if w["next_1w_positive"] == 1)
    return {"query_week": str(rowq["week_id"]), "similar_weeks": weeks,
            "summary": {"pct_followed_by_up": up / max(len(weeks), 1),
                        "avg_next_1w_return": float(np.mean([w["next_1w_return"]
                                              for w in weeks if w["next_1w_return"] is not None]))}}

import json
print(json.dumps(find_similar_regimes(), indent=2, default=str))

Interpret it as a historical analogue, not a prediction. "In 4 of 5 similar weeks the market
rose the following week" is context, not certainty.